# Usecase 3: CRC classification data preparation

Prepares the dataset for the binary classification usecase following [Topcuoğlu et al. 2020](https://doi.org/10.1128/mBio.00434-20). Target: screen-relevant neoplasias (SRN: advanced adenoma + carcinoma) vs. healthy colon, on the Baxter et al. 2016 cohort.

Both feature tables ship pre-processed in the SchlossLab GitHub repos, so no mothur run is required:
- **Un-rarefied OTU table** (`baxter.shared`, 11,282 OTUs × 490 samples) → consumed by *ritme*
- **Rarefied/subsampled OTU table** (`baxter.0.03.subsample.shared`, 6,920 OTUs × 490 samples at 10,423 reads each) → consumed by the original comparator

This mirrors the U1 pattern where ritme sees the raw clustered counts and the original baseline sees the publication's own preprocessed table.

This notebook can be run in the following conda environment (the last command must be launched from the root of this repository):
```shell
mamba create -n ritme_usecases -c adamova -c conda-forge -c bioconda -c pytorch ritme ipykernel nbconvert -y
conda activate ritme_usecases
pip install -e .
```

## Setup

In [1]:
import os
import re
import subprocess

import numpy as np
import pandas as pd

%load_ext autoreload
%autoreload 2

In [2]:
######## USER INPUTS ########
path_to_data = "../../data/u3_topcuoglu20_baxter"

# Source URLs (SchlossLab repos — both tables and metadata ship pre-processed).
URL_UNRAREFIED = (
    "https://raw.githubusercontent.com/SchlossLab/Sze_CRCMetaAnalysis_mBio_2018"
    "/master/data/process/baxter/baxter.shared"
)
URL_SUBSAMPLED = (
    "https://raw.githubusercontent.com/SchlossLab/Topcuoglu_ML_mBio_2020"
    "/master/data/baxter.0.03.subsample.shared"
)
URL_TAXONOMY = (
    "https://raw.githubusercontent.com/SchlossLab/Sze_CRCMetaAnalysis_mBio_2018"
    "/master/data/process/baxter/baxter.taxonomy"
)
URL_METADATA = (
    "https://raw.githubusercontent.com/SchlossLab/Topcuoglu_ML_mBio_2020"
    "/master/data/metadata.tsv"
)
######## END USER INPUTS #####

## Fetch data

In [3]:
os.makedirs(path_to_data, exist_ok=True)

downloads = {
    "baxter.shared": URL_UNRAREFIED,
    "baxter.0.03.subsample.shared": URL_SUBSAMPLED,
    "baxter.taxonomy": URL_TAXONOMY,
    "metadata.tsv": URL_METADATA,
}
for fname, url in downloads.items():
    dst = os.path.join(path_to_data, fname)
    if os.path.exists(dst):
        print(f"[skip] {dst} already exists")
        continue
    print(f"[fetch] {url} -> {dst}")
    subprocess.run(["curl", "-fLso", dst, url], check=True)

[skip] ../../data/u3_topcuoglu20_baxter/baxter.shared already exists
[skip] ../../data/u3_topcuoglu20_baxter/baxter.0.03.subsample.shared already exists
[skip] ../../data/u3_topcuoglu20_baxter/baxter.taxonomy already exists
[skip] ../../data/u3_topcuoglu20_baxter/metadata.tsv already exists


## Parse the two `.shared` feature tables

mothur's `.shared` format: `label \t Group \t numOtus \t Otu00001 \t ...`.

In [4]:
def load_shared(path):
    df = pd.read_csv(path, sep="\t")
    df = df.drop(columns=["label", "numOtus"])
    df = df.rename(columns={"Group": "sample_id"})
    df["sample_id"] = df["sample_id"].astype(str)
    return df.set_index("sample_id")


ft_unrarefied = load_shared(os.path.join(path_to_data, "baxter.shared"))
ft_subsampled = load_shared(os.path.join(path_to_data, "baxter.0.03.subsample.shared"))
print(
    "un-rarefied:",
    ft_unrarefied.shape,
    "row sum range:",
    ft_unrarefied.sum(axis=1).min(),
    ft_unrarefied.sum(axis=1).max(),
)
print(
    "subsampled :",
    ft_subsampled.shape,
    "row sum range:",
    ft_subsampled.sum(axis=1).min(),
    ft_subsampled.sum(axis=1).max(),
)

un-rarefied: (490, 11282) row sum range: 10423 423004
subsampled : (490, 6920) row sum range: 10423 10423


In [5]:
ft_unrarefied.to_csv(
    os.path.join(path_to_data, "otu_table_baxter_unrarefied.tsv"), sep="\t"
)
ft_subsampled.to_csv(
    os.path.join(path_to_data, "otu_table_baxter_subsampled.tsv"), sep="\t"
)

## Process taxonomy

mothur's `.taxonomy` format: `OTU \t Size \t Taxonomy` with bootstrap-support parens (e.g. `Bacteria(100);Firmicutes(95);...`).

In [6]:
# mothur's `.taxonomy` is fixed 6-level (K;P;C;O;F;G) for 16S OTUs at 97%
# similarity. Add Greengenes-style rank prefixes so ritme's taxonomic
# aggregation feature engineering can identify each rank — matching the
# format produced by U1/U2/U3 (k__ p__ c__ o__ f__ g__). Bootstrap-support
# parens are stripped first; mothur's `<parent>_unclassified` markers are
# kept as-is (informative for aggregation).
RANK_PREFIXES = ["k__", "p__", "c__", "o__", "f__", "g__"]


def _format_mothur_taxon(taxonomy_str):
    levels = [
        t.strip() for t in re.sub(r"\(\d+\)", "", taxonomy_str).strip(";").split(";")
    ]
    if len(levels) != len(RANK_PREFIXES):
        raise ValueError(
            f"Expected {len(RANK_PREFIXES)}-level mothur taxonomy; got "
            f"{len(levels)}: {levels}"
        )
    return "; ".join(f"{p}{lvl}" for p, lvl in zip(RANK_PREFIXES, levels))


tax_raw = pd.read_csv(os.path.join(path_to_data, "baxter.taxonomy"), sep="\t")
tax_raw["Taxon"] = tax_raw["Taxonomy"].apply(_format_mothur_taxon)
tax_df = tax_raw.set_index("OTU")[["Taxon"]]
tax_df.index.name = "Feature ID"
tax_df.to_csv(os.path.join(path_to_data, "taxonomy_baxter.tsv"), sep="\t")
print(tax_df.shape)
tax_df.head()

(11282, 1)


,Taxon
Feature ID,
Otu00001,k__Bacteria; p__Firmicutes; c__Clostridia; o__...
Otu00002,k__Bacteria; p__Bacteroidetes; c__Bacteroidia;...
Otu00003,k__Bacteria; p__Bacteroidetes; c__Bacteroidia;...
Otu00004,k__Bacteria; p__Verrucomicrobia; c__Verrucomic...
Otu00005,k__Bacteria; p__Bacteroidetes; c__Bacteroidia;...


## Process metadata

Derive the binary `srn` label from `Dx_Bin` per Topcuoğlu's definition (SRN = advanced adenoma or carcinoma; healthy class includes non-advanced adenomas, normal colons, and high-risk normals). Normalize column names to lowercase snake_case so they round-trip cleanly through ritme's `data_enrich_with` list.

In [7]:
md_raw = pd.read_csv(os.path.join(path_to_data, "metadata.tsv"), sep="\t")
md_raw["sample"] = md_raw["sample"].astype(str)
md_raw = md_raw.set_index("sample")
md_raw.index.name = "sample_id"

# Fail loudly if SchlossLab changes the Dx_Bin universe (renames, new
# buckets, capitalization shifts). Without this assert, an unknown value
# would silently land in the negative class and corrupt the SRN label.
KNOWN_DX_BIN = {"Normal", "High Risk Normal", "Adenoma", "adv Adenoma", "Cancer"}
observed_dx_bin = set(md_raw["Dx_Bin"].unique())
unknown = observed_dx_bin - KNOWN_DX_BIN
assert not unknown, (
    f"Unknown Dx_Bin values: {unknown}. Update KNOWN_DX_BIN and the SRN "
    "rule before re-running."
)

# Binary SRN per Topcuoğlu 2020: advanced adenoma + cancer = 1; the
# healthy class (= non-advanced adenomas + normal colons + high-risk
# normals) = 0.
md_raw["srn"] = md_raw["Dx_Bin"].isin(["adv Adenoma", "Cancer"]).astype(int)

# Lowercase snake_case so config strings stay portable.
md_raw.columns = [c.lower() for c in md_raw.columns]

# fit_result is a numeric count; ensure dtype.
md_raw["fit_result"] = pd.to_numeric(md_raw["fit_result"], errors="coerce")

md_raw["srn"].value_counts()

srn
0    261
1    229
Name: count, dtype: int64

In [8]:
md_raw.to_csv(os.path.join(path_to_data, "md_baxter.tsv"), sep="\t")
print(md_raw.shape)
md_raw.head()

(490, 27)


,fit_result,site,dx_bin,dx,hx_prev,hx_of_polyps,age,gender,smoke,diabetic,...,pacific,asian,other,ethnic,nsaid,abx,diabetes_med,stage,location,srn
sample_id,,,,,,,,,,,,,,,,,,,,,
2003650,0,U Michigan,High Risk Normal,normal,0.0,1.0,64,m,NaN,0.0,...,0,0,0,0.0,0.0,0,0,0,NaN,0
2005650,0,U Michigan,High Risk Normal,normal,0.0,1.0,61,m,0.0,0.0,...,0,0,0,0.0,0.0,0,0,0,NaN,0
2007660,26,U Michigan,High Risk Normal,normal,0.0,1.0,47,f,0.0,0.0,...,0,0,0,0.0,0.0,0,0,0,NaN,0
2009650,10,Toronto,Adenoma,adenoma,0.0,1.0,81,f,1.0,0.0,...,0,0,0,0.0,1.0,0,0,0,NaN,0
2013660,0,U Michigan,Normal,normal,0.0,0.0,44,f,0.0,0.0,...,0,0,0,0.0,1.0,0,0,0,NaN,0


## Sanity checks

In [9]:
assert ft_unrarefied.shape[0] == 490, ft_unrarefied.shape
assert ft_unrarefied.shape[1] == 11282, ft_unrarefied.shape
assert ft_subsampled.shape == (490, 6920), ft_subsampled.shape
assert (
    ft_subsampled.sum(axis=1) == 10423
).all(), "subsampled rows should all sum to 10423"
assert int(md_raw["srn"].sum()) == 229, md_raw["srn"].value_counts().to_dict()
assert int((md_raw["srn"] == 0).sum()) == 261
assert set(md_raw.index) == set(ft_unrarefied.index) == set(ft_subsampled.index)
print("All sanity checks passed.")

All sanity checks passed.
